In [1]:
import os
import pandas as pd
import pyarrow

In [2]:
os.chdir('../data')

In [3]:
df = pd.read_csv('flight_data_2024.csv')
df.info()

/tmp/ipykernel_24420/3195983607.py:1: DtypeWarning: Columns (0: cancellation_code) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('flight_data_2024.csv')


<class 'pandas.DataFrame'>
RangeIndex: 7079081 entries, 0 to 7079080
Data columns (total 35 columns):
 #   Column               Dtype  
---  ------               -----  
 0   year                 int64  
 1   month                int64  
 2   day_of_month         int64  
 3   day_of_week          int64  
 4   fl_date              str    
 5   op_unique_carrier    str    
 6   op_carrier_fl_num    float64
 7   origin               str    
 8   origin_city_name     str    
 9   origin_state_nm      str    
 10  dest                 str    
 11  dest_city_name       str    
 12  dest_state_nm        str    
 13  crs_dep_time         int64  
 14  dep_time             float64
 15  dep_delay            float64
 16  taxi_out             float64
 17  wheels_off           float64
 18  wheels_on            float64
 19  taxi_in              float64
 20  crs_arr_time         int64  
 21  arr_time             float64
 22  arr_delay            float64
 23  cancelled            int64  
 24  cancellat

In [4]:
df.head(10)

,year,month,day_of_month,day_of_week,fl_date,op_unique_carrier,op_carrier_fl_num,origin,origin_city_name,origin_state_nm,...,diverted,crs_elapsed_time,actual_elapsed_time,air_time,distance,carrier_delay,weather_delay,nas_delay,security_delay,late_aircraft_delay
0,2024,1,1,1,2024-01-01,9E,4814.0,JFK,"New York, NY",New York,...,0,136.0,122.0,84.0,509.0,0,0,0,0,0
1,2024,1,1,1,2024-01-01,9E,4815.0,MSP,"Minneapolis, MN",Minnesota,...,0,130.0,114.0,88.0,622.0,0,0,0,0,0
2,2024,1,1,1,2024-01-01,9E,4817.0,JFK,"New York, NY",New York,...,0,106.0,90.0,61.0,288.0,0,0,0,0,0
3,2024,1,1,1,2024-01-01,9E,4817.0,RIC,"Richmond, VA",Virginia,...,0,111.0,76.0,51.0,288.0,0,0,0,0,0
4,2024,1,1,1,2024-01-01,9E,4818.0,DTW,"Detroit, MI",Michigan,...,0,79.0,70.0,45.0,237.0,0,0,0,0,0
5,2024,1,1,1,2024-01-01,9E,4822.0,JAX,"Jacksonville, FL",Florida,...,0,137.0,120.0,102.0,833.0,0,0,0,0,0
6,2024,1,1,1,2024-01-01,9E,4822.0,LGA,"New York, NY",New York,...,0,169.0,164.0,125.0,833.0,0,0,0,0,0
7,2024,1,1,1,2024-01-01,9E,4823.0,CHS,"Charleston, SC",South Carolina,...,0,118.0,99.0,86.0,641.0,0,0,0,0,0
8,2024,1,1,1,2024-01-01,9E,4823.0,LGA,"New York, NY",New York,...,0,149.0,123.0,101.0,641.0,0,0,0,0,0
9,2024,1,1,1,2024-01-01,9E,4828.0,ITH,"Ithaca/Cortland, NY",New York,...,0,79.0,67.0,43.0,189.0,0,0,0,0,0


In [5]:
print(f"Record iniziali: {len(df)}")

Record iniziali: 7079081


**Pulizia dei voli cancellati e deviati**

In [6]:
print(df['cancelled'].unique())
print(df['cancelled'].isnull().sum())

[0 1]
0


In [7]:
print(df['diverted'].unique())
print(df['diverted'].isnull().sum())

[0 1]
0


Tutto bene, non ci sono valori nulli o valori sporchi

Adesso si crea una maschera booleana, per identificare gli eventuali record corrotti: voli regolari ma con ritardi mancanti

In [8]:
maschere = (df['cancelled'] == 0) & (df['diverted'] == 0) & (df['arr_delay'].isna() | df['dep_delay'].isna())
df_clean = df[~maschere].copy()
print(f"Record dopo la prima pulizia: {len(df_clean)}")

Record dopo la prima pulizia: 7079081


In [9]:
print(df['year'].unique())

[2024]


Cancello la colonna year: contiene sempre lo stesso campo, perciò è del tutto superfluo

**Parsing della data**

In [10]:
df_clean = df_clean.drop(columns=['year'])

In [11]:
df_clean['fl_date'] = pd.to_datetime(df_clean['fl_date'])

**Selezione delle colonne target**

In [12]:
target = ['month', 'fl_date', 
    'op_unique_carrier', 'op_carrier_fl_num', 
    'origin', 'dest', 
    'dep_delay', 'arr_delay', 
    'cancelled', 'diverted', 'cancellation_code',
    'carrier_delay', 'weather_delay', 'nas_delay', 'security_delay', 'late_aircraft_delay']

# Per evitare errori
esistenti = [colonna for colonna in target if colonna in df_clean.columns]
df_clean = df_clean[esistenti]

**Normalizzazione e imputazione per le cause di ritardo**

Un NaN in queste colonne significa "0 minuti di ritardo imputabili a questa causa"

In [13]:
cause = ['carrier_delay', 'weather_delay', 'nas_delay', 'security_delay', 'late_aircraft_delay']
for causa in cause:
    if causa in df_clean.columns:
        df_clean[causa] = pd.to_numeric(df_clean[causa], errors='coerce').fillna(0)

In [14]:
print(f"Numero di colonne mantenute: {len(df_clean.columns)}")

Numero di colonne mantenute: 16


In [15]:
df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 7079081 entries, 0 to 7079080
Data columns (total 16 columns):
 #   Column               Dtype         
---  ------               -----         
 0   month                int64         
 1   fl_date              datetime64[us]
 2   op_unique_carrier    str           
 3   op_carrier_fl_num    float64       
 4   origin               str           
 5   dest                 str           
 6   dep_delay            float64       
 7   arr_delay            float64       
 8   cancelled            int64         
 9   diverted             int64         
 10  cancellation_code    str           
 11  carrier_delay        int64         
 12  weather_delay        int64         
 13  nas_delay            int64         
 14  security_delay       int64         
 15  late_aircraft_delay  int64         
dtypes: datetime64[us](1), float64(3), int64(8), str(4)
memory usage: 919.1 MB


In [16]:
df_clean.head(10)

,month,fl_date,op_unique_carrier,op_carrier_fl_num,origin,dest,dep_delay,arr_delay,cancelled,diverted,cancellation_code,carrier_delay,weather_delay,nas_delay,security_delay,late_aircraft_delay
0,1,2024-01-01,9E,4814.0,JFK,DTW,-5.0,-19.0,0,0,NaN,0,0,0,0,0
1,1,2024-01-01,9E,4815.0,MSP,CLE,-14.0,-30.0,0,0,NaN,0,0,0,0,0
2,1,2024-01-01,9E,4817.0,JFK,RIC,-4.0,-20.0,0,0,NaN,0,0,0,0,0
3,1,2024-01-01,9E,4817.0,RIC,JFK,-7.0,-42.0,0,0,NaN,0,0,0,0,0
4,1,2024-01-01,9E,4818.0,DTW,MKE,-5.0,-14.0,0,0,NaN,0,0,0,0,0
5,1,2024-01-01,9E,4822.0,JAX,LGA,-7.0,-24.0,0,0,NaN,0,0,0,0,0
6,1,2024-01-01,9E,4822.0,LGA,JAX,-8.0,-13.0,0,0,NaN,0,0,0,0,0
7,1,2024-01-01,9E,4823.0,CHS,LGA,-5.0,-24.0,0,0,NaN,0,0,0,0,0
8,1,2024-01-01,9E,4823.0,LGA,CHS,-5.0,-31.0,0,0,NaN,0,0,0,0,0
9,1,2024-01-01,9E,4828.0,ITH,JFK,-12.0,-24.0,0,0,NaN,0,0,0,0,0


In [18]:
output_path = 'dataset.parquet'
df_clean.to_parquet(output_path, index=False)
